# Web-Gold-40K RETRY/ABORT supplement v2 sanity

Run this notebook before any new mini/full training. It does not modify the original Gold files, does not read the locked test split, and does not start an epoch. It validates the Kaggle-mounted supplement (already extracted or retained as a ZIP), audits multi-source loading, and performs one T4 forward/backward pass proving that supplement rows supervise only recovery strategy and recovery success.

In [ ]:
# 1. Reject the wrong accelerator before doing any work.
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
assert 'T4' in GPU_NAME.upper(), f'Tesla T4 required; stop this session (found {GPU_NAME}).'

In [ ]:
# 2. Pull the Code branch and install the same pinned runtime as v2.7.
from pathlib import Path
import importlib.metadata as metadata
import json, os, subprocess, sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
requirements = ['transformers>=4.49,<5', 'peft>=0.14,<1', 'bitsandbytes>=0.45,<1', 'accelerate>=1,<2', 'scikit-learn>=1.4,<2']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('commit:', COMMIT)

In [ ]:
# 3. Locate both attached datasets. Kaggle may mount the supplement already extracted.
import hashlib
import zipfile

ORIGINAL_ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
SUPPLEMENT_ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/gold-40k-retry')
SPLIT_FILES = ('split_train.json', 'split_val.json')  # test is deliberately excluded
ARCHIVE_NAME = 'web_gold_40k_retry_abort_supplement_v2_kaggle.zip'
EXPECTED_ARCHIVE_SHA256 = 'c82fd5cb3567ffcbe7c3bf664a3c0aa9b0f4a408f733d321366e1343798b8c02'

def find_original_root(root: Path) -> Path:
    candidates = []
    if root.is_dir() and all((root / name).is_file() for name in SPLIT_FILES):
        candidates.append(root)
    if root.is_dir():
        for current, _, files in os.walk(root, followlinks=True):
            if set(SPLIT_FILES).issubset(files):
                candidates.append(Path(current))
    candidates = sorted(set(path.resolve() for path in candidates))
    assert len(candidates) == 1, f'Attach Web-Gold-40K; expected one train/val split root, found {candidates}'
    return candidates[0]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

ORIGINAL_ROOT = find_original_root(ORIGINAL_ATTACHED_ROOT)
assert SUPPLEMENT_ATTACHED_ROOT.is_dir(), f'Attach the supplement dataset at {SUPPLEMENT_ATTACHED_ROOT}'
archives = sorted(SUPPLEMENT_ATTACHED_ROOT.rglob(ARCHIVE_NAME))
mounted_train_files = sorted(
    path for path in SUPPLEMENT_ATTACHED_ROOT.rglob('supplement_train.json')
    if path.parent.name == 'data'
)
assert len(archives) <= 1, f'Expected at most one supplement ZIP; found {archives}'
assert len(mounted_train_files) <= 1, f'Expected at most one mounted supplement root; found {mounted_train_files}'
SUPPLEMENT_ARCHIVE = archives[0].resolve() if archives else None
if SUPPLEMENT_ARCHIVE is not None:
    actual_sha = sha256_file(SUPPLEMENT_ARCHIVE)
    assert actual_sha == EXPECTED_ARCHIVE_SHA256, f'Supplement ZIP hash mismatch: {actual_sha}'

if mounted_train_files:
    # Normal Kaggle dataset behavior: the uploaded ZIP is unpacked at mount time.
    SUPPLEMENT_ROOT = mounted_train_files[0].parent.parent.resolve()
else:
    assert SUPPLEMENT_ARCHIVE is not None, (
        f'No data/supplement_train.json or {ARCHIVE_NAME} found under '
        f'{SUPPLEMENT_ATTACHED_ROOT}'
    )
    EXTRACT_ROOT = Path('/kaggle/working/retry_abort_supplement_v2_extracted')
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SUPPLEMENT_ARCHIVE) as archive:
        destination = EXTRACT_ROOT.resolve()
        for member in archive.infolist():
            candidate = (destination / member.filename).resolve()
            assert candidate == destination or destination in candidate.parents, f'Unsafe ZIP path: {member.filename}'
        archive.extractall(destination)
    extracted = sorted(EXTRACT_ROOT.rglob('data/supplement_train.json'))
    assert len(extracted) == 1, f'Expected one extracted supplement root; found {extracted}'
    SUPPLEMENT_ROOT = extracted[0].parent.parent.resolve()
print('ORIGINAL_ROOT =', ORIGINAL_ROOT)
print('SUPPLEMENT_ARCHIVE =', SUPPLEMENT_ARCHIVE or 'Kaggle mounted extracted files')
print('SUPPLEMENT_ROOT =', SUPPLEMENT_ROOT)

In [ ]:
# 4. Validate archive/package integrity, exact counts, images, review state, and split isolation.
PACKAGE_REPORT = Path('/kaggle/working/retry_abort_supplement_v2_validation_report.json')
command = [
    sys.executable, 'scripts/validate_retry_abort_supplement.py',
    '--supplement-root', str(SUPPLEMENT_ROOT),
    '--report', str(PACKAGE_REPORT),
]
if SUPPLEMENT_ARCHIVE is not None:
    command.extend(['--archive', str(SUPPLEMENT_ARCHIVE)])
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
package_report = json.loads(PACKAGE_REPORT.read_text(encoding='utf-8'))
assert package_report['status'] == 'PASS'
assert package_report['test_rows_read'] == 0

In [ ]:
# 5. Prove multi-source counts, source tags, masks, and original-only primary validation.
MULTISOURCE_REPORT = Path('/kaggle/working/retry_abort_supplement_v2_multisource_report.json')
command = [
    sys.executable, 'scripts/audit_retry_abort_multisource.py',
    '--original-root', str(ORIGINAL_ROOT),
    '--supplement-root', str(SUPPLEMENT_ROOT),
    '--report', str(MULTISOURCE_REPORT),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
multisource_report = json.loads(MULTISOURCE_REPORT.read_text(encoding='utf-8'))
assert multisource_report['status'] == 'PASS'
assert multisource_report['primary_training_rows'] == 24107
assert multisource_report['primary_validation_rows'] == 7861
assert multisource_report['supplement_validation_rows_available_separately'] == 194
assert multisource_report['test_rows_read'] == 0

In [ ]:
# 6. One forward/backward batch: recovery heads must learn; unrelated output heads must not.
SMOKE_REPORT = Path('/kaggle/working/retry_abort_supplement_v2_smoke_report.json')
command = [
    sys.executable, 'scripts/smoke_retry_abort_supplement.py',
    '--supplement-root', str(SUPPLEMENT_ROOT),
    '--report', str(SMOKE_REPORT),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
smoke_report = json.loads(SMOKE_REPORT.read_text(encoding='utf-8'))
assert smoke_report['status'] == 'PASS'
assert smoke_report['test_rows_read'] == 0

In [ ]:
# 7. Final decision. This notebook intentionally stops before mini/full training.
assert package_report['status'] == multisource_report['status'] == smoke_report['status'] == 'PASS'
summary = {
    'status': 'PASS',
    'git_commit': COMMIT,
    'gpu': GPU_NAME,
    'test_rows_read': 0,
    'next_action': 'Run a short controlled v2.8 mini with original and supplement validation reported separately.',
    'reports': [str(PACKAGE_REPORT), str(MULTISOURCE_REPORT), str(SMOKE_REPORT)],
}
SUMMARY_PATH = Path('/kaggle/working/retry_abort_supplement_v2_sanity_summary.json')
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('SANITY PASSED. Do not start full training until the controlled mini is reviewed.')